# 📊 Screener de Ações — Ibovespa

**Objetivo:** Identificar as 20 melhores oportunidades do Ibovespa (ações + bancos) que atendam a critérios fundamentalistas rigorosos e estejam com preço de mercado abaixo do preço justo.

**Critérios de Screening (Ações — 11 critérios):**
| # | Critério | Valor |
|---|----------|-------|
| 1 | P/L | entre 0 e 10 |
| 2 | P/PV | entre 0 e 1,5 |
| 3 | Margem EBIT | positiva |
| 4 | Margem Líquida | > 10% |
| 5 | Dívida Líq./EBIT | < 3 |
| 6 | Dívida Líq./PL | < 2 |
| 7 | ROE | > 10% |
| 8 | Liquidez Corrente | > 1 |
| 9 | Passivos/Ativos | < 1 |
| 10 | Liq. Média Diária | > R$ 100 mil |
| 11 | LPA | > 0 |

**Critérios de Screening (Bancos — 7 critérios):**
P/L 0–10, P/PV 0–2, ROE > 15%, Margem Líq. > 10%, LPA > 0, Liq. Diária > R$ 100k, DY > 3%.

**Valuation (inspirado Simply Wall St):**
- **Ações:** DCF 2-estágios (decay linear, 10 anos) + DDM fallback + Graham
- **Bancos:** Excess Returns Model + Graham
- ⭐ **Forte desconto** = margem de segurança média ≥ 20%

In [14]:
# === Setup ===
import sys
import importlib
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.width', 200)

# Garantir que o diretório raiz está no path para importar src/
if '.' not in sys.path:
    sys.path.insert(0, '.')

from src import scraper, fundamentals, filters, valuation

# Recarregar módulos durante desenvolvimento
for mod in [scraper, fundamentals, filters, valuation]:
    importlib.reload(mod)

print("✅ Módulos carregados com sucesso")

✅ Módulos carregados com sucesso


## 1. Coleta de Tickers

Scraping de `dadosdemercado.com.br/acoes` para obter a lista completa de ações brasileiras, seguido de classificação em **bancos** e **não-bancos**.

In [15]:
# Obter tickers de ações brasileiras
tickers_df = scraper.get_tickers()
print(f"Total de tickers: {len(tickers_df)}")
tickers_df.head(10)

[scraper] 388 tickers carregados do cache (/Users/alisson/Documents/projects/personal/stock-analysis/data/tickers.csv)
Total de tickers: 388


,ticker,ticker_sa
0,PETR4,PETR4.SA
1,BBDC4,BBDC4.SA
2,B3SA3,B3SA3.SA
3,ITSA4,ITSA4.SA
4,GMAT3,GMAT3.SA
5,PGMN3,PGMN3.SA
6,AZEV4,AZEV4.SA
7,COGN3,COGN3.SA
8,VAMO3,VAMO3.SA
9,VALE3,VALE3.SA


## 2. Coleta de Dados Fundamentalistas

Busca de métricas via `yfinance` para todas as ações. Isso pode levar alguns minutos dependendo do número de tickers.

In [16]:
# Coletar fundamentals para todas as ações
stock_fundamentals = fundamentals.fetch_fundamentals(tickers_df['ticker_sa'].tolist(), delay=0.4)
stock_fundamentals.head()

[fundamentals] 388 tickers carregados do cache (/Users/alisson/Documents/projects/personal/stock-analysis/data/fundamentals.csv)


,ticker_sa,ticker,nome,setor,industria,preco,pl,pvp,margem_ebit_pct,margem_liquida_pct,dl_ebit,dl_pl,roe_pct,liquidez_corrente,passivos_ativos,liq_media_diaria,lpa,vpa,dy_pct,divida_liquida,ebit,fcf_latest,shares_outstanding,dividend_rate
0,PETR4.SA,PETR4,PETROBRAS PN ATZ N2,Energy,Oil & Gas Integrated,48.70,6.02,1.51,30.41,22.13,12.07,4.39,28.18,0.71,0.66,3103472758.00,8.10,32.24,8.25,333417000960.00,27614797780.70,16721236676.89,5446501379.00,3.91
1,BBDC4.SA,BBDC4,BRADESCO PN N1,Financial Services,Banks - Regional,18.95,8.90,1.12,NaN,26.52,NaN,3.06,13.75,NaN,0.92,691219442.00,2.13,16.87,1.17,545811021824.00,NaN,-74164468000.00,5277491247.00,0.23
2,B3SA3.SA,B3SA3,B3 ON NM,Financial Services,Financial Data & Stock Exchanges,17.99,21.42,5.19,86.09,45.54,0.13,0.06,25.59,1.91,0.64,747792668.70,0.84,3.46,3.27,1114740736.00,8666644000.00,4071357000.00,5010732147.00,0.61
3,ITSA4.SA,ITSA4,ITAUSA PN EJ N1,Industrials,Conglomerates,13.86,9.36,1.75,216.77,199.87,0.35,0.07,17.57,3.18,0.16,476384970.60,1.48,7.92,68.00,6233999872.00,17881000000.00,-765000000.00,7357712522.00,0.10
4,GMAT3.SA,GMAT3,GRUPO MATEUSON NM,Consumer Cyclical,Department Stores,4.53,5.52,0.98,4.52,4.77,2.86,0.48,17.33,1.75,0.53,109891684.50,0.82,4.63,5.35,4978662016.00,1738664000.00,814496000.00,2300047621.00,0.25


## 3. Screening — Ações (não-bancos)

Aplicação dos 11 critérios fundamentalistas.

In [17]:
# Aplicar filtros fundamentalistas para ações
# Sanitizar colunas numéricas para evitar comparação str vs int nos filtros
numeric_cols = [
    'preco', 'pl', 'pvp', 'margem_ebit_pct', 'margem_liquida_pct',
    'dl_ebit', 'dl_pl', 'roe_pct', 'liquidez_corrente', 'passivos_ativos',
    'liq_media_diaria', 'lpa', 'vpa', 'dy_pct', 'divida_liquida', 'ebit',
    'fcf_latest', 'shares_outstanding', 'dividend_rate'
]

stock_fundamentals_clean = stock_fundamentals.copy()
for col in numeric_cols:
    if col in stock_fundamentals_clean.columns:
        stock_fundamentals_clean[col] = pd.to_numeric(
            stock_fundamentals_clean[col], errors='coerce'
        )

filtered_stocks = filters.apply_stock_filters(stock_fundamentals_clean)

if len(filtered_stocks) > 0:
    display_cols = ['ticker', 'nome', 'setor', 'preco', 'pl', 'pvp',
                    'margem_ebit_pct', 'margem_liquida_pct', 'dl_ebit', 'dl_pl',
                    'roe_pct', 'liquidez_corrente', 'passivos_ativos', 'lpa']
    display(filtered_stocks[display_cols].style.format({
        'preco': 'R$ {:.2f}', 'pl': '{:.2f}', 'pvp': '{:.2f}',
        'margem_ebit_pct': '{:.1f}%', 'margem_liquida_pct': '{:.1f}%',
        'dl_ebit': '{:.2f}', 'dl_pl': '{:.2f}', 'roe_pct': '{:.1f}%',
        'liquidez_corrente': '{:.2f}', 'passivos_ativos': '{:.2f}', 'lpa': 'R$ {:.2f}',
    }))
else:
    print("⚠️ Nenhuma ação passou em todos os 11 critérios."
          "\nIsso pode ocorrer em mercados sobrevalorizados."
          "\nConsidere relaxar alguns critérios.")

[filters] Ações: 27/388 passaram nos critérios


,ticker,nome,setor,preco,pl,pvp,margem_ebit_pct,margem_liquida_pct,dl_ebit,dl_pl,roe_pct,liquidez_corrente,passivos_ativos,lpa
0,CMIG4,CEMIG PN EJ N1,Utilities,R$ 12.52,7.32,1.25,17.0%,11.5%,2.40,0.60,17.5%,1.00,0.57,R$ 1.71
1,CYRE3,CYRELA REALTON NM,Consumer Cyclical,R$ 26.84,5.87,0.96,33.6%,21.2%,1.30,0.40,22.4%,3.84,0.56,R$ 4.57
2,GRND3,GRENDENE ON NM,Consumer Cyclical,R$ 4.62,6.51,1.32,28.9%,25.0%,-1.43,-0.31,17.9%,2.05,0.30,R$ 0.71
3,JHSF3,JHSF PART ON NM,Real Estate,R$ 9.38,3.39,0.99,75.8%,66.0%,1.40,0.55,21.0%,2.80,0.62,R$ 2.77
4,SUZB3,SUZANO S.A. ON NM,Basic Materials,R$ 50.91,4.71,1.44,53.0%,26.8%,2.88,1.75,35.2%,3.19,0.74,R$ 10.81
5,RECV3,PETRORECSA ON NM,Energy,R$ 14.09,6.46,0.95,34.1%,20.2%,1.40,0.35,14.9%,3.48,0.50,R$ 2.18
6,VTRU3,VITRUEDUCA ON NM,Consumer Defensive,R$ 14.31,5.61,0.65,29.2%,16.4%,2.98,0.67,13.5%,2.23,0.52,R$ 2.55
7,EZTC3,EZTEC ON NM,Real Estate,R$ 13.79,7.26,0.76,45.6%,35.7%,0.25,0.03,10.9%,8.33,0.29,R$ 1.90
8,EVEN3,EVEN ON NM,Real Estate,R$ 7.00,4.93,0.73,20.1%,12.4%,1.33,0.27,14.0%,4.79,0.58,R$ 1.42
9,SAPR4,SANEPAR PN N2,Utilities,R$ 8.65,6.71,0.21,40.3%,28.9%,0.62,0.14,17.9%,1.20,0.53,R$ 1.29


## 4. Valuation — Preço Justo

Cálculo do preço justo de cada ação filtrada por dois métodos:
1. **DCF 2-Estágios** — Crescimento decai linearmente de CAGR histórico → inflação LP (3,5%) ao longo de 10 anos. Fallback: DDM quando FCF indisponível.
2. **Fórmula de Graham** — `V = √(P/L_setor × P/PV_setor × LPA × VPA)`

Para **bancos**, o modelo primário é **Excess Returns**: `FV = VPA + (ROE - CoE) × VPA / (CoE - g)`

Ações com preço de mercado **abaixo de ambos** os preços justos são consideradas subvalorizadas.
⭐ **Forte desconto** = margem de segurança média ≥ 20% (inspirado Simply Wall St).

In [18]:
# Calcular valuation (DCF 2-estágios + Graham) para ações filtradas
if len(filtered_stocks) > 0:
    valued_stocks = valuation.apply_valuation(filtered_stocks, stock_fundamentals_clean, model='stock')

    val_cols = ['ticker', 'nome', 'preco', 'preco_justo_dcf', 'preco_justo_graham',
                'margem_seg_dcf_pct', 'margem_seg_graham_pct', 'undervalued', 'forte_desconto']
    display(valued_stocks[val_cols].style.format({
        'preco': 'R$ {:.2f}',
        'preco_justo_dcf': 'R$ {:.2f}',
        'preco_justo_graham': 'R$ {:.2f}',
        'margem_seg_dcf_pct': '{:.1f}%',
        'margem_seg_graham_pct': '{:.1f}%',
    }))
else:
    valued_stocks = pd.DataFrame()
    print("⚠️ Nenhuma ação para avaliar.")

[valuation] 19/27 ações abaixo do preço justo | 16 com forte desconto (≥20%)


,ticker,nome,preco,preco_justo_dcf,preco_justo_graham,margem_seg_dcf_pct,margem_seg_graham_pct,undervalued,forte_desconto
0,CMIG4,CEMIG PN EJ N1,R$ 12.52,R$ 13.27,R$ 16.59,5.6%,24.5%,True,False
1,CYRE3,CYRELA REALTON NM,R$ 26.84,R$ 35.35,R$ 37.54,24.1%,28.5%,True,True
2,GRND3,GRENDENE ON NM,R$ 4.62,R$ 4.79,R$ 5.24,3.5%,11.8%,True,False
3,JHSF3,JHSF PART ON NM,R$ 9.38,R$ 34.73,R$ 13.30,73.0%,29.5%,True,True
4,SUZB3,SUZANO S.A. ON NM,R$ 50.91,R$ 38.01,R$ 80.01,-33.9%,36.4%,False,False
5,RECV3,PETRORECSA ON NM,R$ 14.09,R$ 17.86,R$ 18.49,21.1%,23.8%,True,True
6,VTRU3,VITRUEDUCA ON NM,R$ 14.31,R$ 32.78,R$ 30.58,56.3%,53.2%,True,True
7,EZTC3,EZTEC ON NM,R$ 13.79,R$ 14.64,R$ 15.28,5.8%,9.7%,True,False
8,EVEN3,EVEN ON NM,R$ 7.00,R$ 3.82,R$ 9.60,-83.1%,27.1%,False,False
9,SAPR4,SANEPAR PN N2,R$ 8.65,R$ 78.38,R$ 29.14,89.0%,70.3%,True,True


## 4.1. Screening + Valuation — Bancos

Filtros adaptados para bancos (7 critérios) + valuation via **Excess Returns Model** e **Graham**.

In [19]:
# Separar bancos dos dados fundamentalistas usando a classificação do scraper
from src.scraper import KNOWN_BANK_TICKERS, BANK_INDUSTRIES

bank_mask = (
    stock_fundamentals_clean['ticker'].isin(KNOWN_BANK_TICKERS) |
    stock_fundamentals_clean['industria'].str.lower().isin(BANK_INDUSTRIES)
)
bank_fundamentals = stock_fundamentals_clean[bank_mask].copy()
print(f"Bancos identificados: {len(bank_fundamentals)}")

# Aplicar filtros para bancos
filtered_banks = filters.apply_bank_filters(bank_fundamentals)

# Calcular valuation (Excess Returns + Graham) para bancos
if len(filtered_banks) > 0:
    valued_banks = valuation.apply_valuation(filtered_banks, stock_fundamentals_clean, model='bank')

    val_cols = ['ticker', 'nome', 'preco', 'preco_justo_dcf', 'preco_justo_graham',
                'margem_seg_dcf_pct', 'margem_seg_graham_pct', 'undervalued', 'forte_desconto']
    display(valued_banks[val_cols].style.format({
        'preco': 'R$ {:.2f}',
        'preco_justo_dcf': 'R$ {:.2f}',
        'preco_justo_graham': 'R$ {:.2f}',
        'margem_seg_dcf_pct': '{:.1f}%',
        'margem_seg_graham_pct': '{:.1f}%',
    }))
else:
    valued_banks = pd.DataFrame()
    print("⚠️ Nenhum banco passou nos critérios de filtragem.")

Bancos identificados: 35
[filters] Bancos: 6/35 passaram nos critérios
[valuation] 4/6 bancos abaixo do preço justo | 4 com forte desconto (≥20%)


,ticker,nome,preco,preco_justo_dcf,preco_justo_graham,margem_seg_dcf_pct,margem_seg_graham_pct,undervalued,forte_desconto
0,ITSA4,ITAUSA PN EJ N1,R$ 13.86,R$ 10.36,R$ 13.52,-33.7%,-2.5%,False,False
1,BRSR6,BANRISUL PNB N1,R$ 17.52,R$ 31.73,R$ 32.35,44.8%,45.8%,True,True
2,ABCB4,ABC BRASIL PN N2,R$ 25.70,R$ 31.19,R$ 36.55,17.6%,29.7%,True,True
3,ITSA3,ITAUSA ON EJ N1,R$ 13.80,R$ 10.36,R$ 13.52,-33.2%,-2.1%,False,False
4,BSLI4,BRB BANCO PN,R$ 3.90,R$ 16.49,R$ 12.15,76.4%,67.9%,True,True
5,BAZA3,AMAZONIA ON,R$ 82.34,R$ 141.87,R$ 155.65,42.0%,47.1%,True,True


## 5. 🏆 Top 20 — Ranking Final (Ações + Bancos)

Ações e bancos subvalorizados (preço < preço justo primário **e** Graham), ordenados por **margem de segurança média**.
⭐ = forte desconto (≥ 20% de margem de segurança).

In [20]:
# Unir ações e bancos valorizados
all_valued = []
if len(valued_stocks) > 0:
    vs = valued_stocks.copy()
    vs['tipo'] = 'ação'
    all_valued.append(vs)
if len(valued_banks) > 0:
    vb = valued_banks.copy()
    vb['tipo'] = 'banco'
    all_valued.append(vb)

if all_valued:
    all_valued_df = pd.concat(all_valued, ignore_index=True)
else:
    all_valued_df = pd.DataFrame()

# Filtrar subvalorizados e rankear
if len(all_valued_df) > 0 and all_valued_df['undervalued'].any():
    undervalued = all_valued_df[all_valued_df['undervalued']].copy()

    # Ordenar por maior margem de segurança e pegar top 20
    top20 = (undervalued
             .sort_values('margem_seg_media_pct', ascending=False)
             .head(20)
             .reset_index(drop=True))

    print(f"🏆 TOP {len(top20)} SUBVALORIZADAS (ações + bancos)\n")

    result_cols = ['tipo', 'ticker', 'nome', 'setor', 'preco', 'preco_justo_dcf',
                   'preco_justo_graham', 'margem_seg_media_pct', 'forte_desconto',
                   'pl', 'pvp', 'roe_pct', 'margem_liquida_pct']
    display(top20[result_cols].style.format({
        'preco': 'R$ {:.2f}',
        'preco_justo_dcf': 'R$ {:.2f}',
        'preco_justo_graham': 'R$ {:.2f}',
        'margem_seg_media_pct': '{:.1f}%',
        'pl': '{:.2f}',
        'pvp': '{:.2f}',
        'roe_pct': '{:.1f}%',
        'margem_liquida_pct': '{:.1f}%',
    }).set_caption("Top 20 — Subvalorizadas por margem de segurança"))
else:
    top20 = pd.DataFrame()
    print("⚠️ Nenhuma ação/banco encontrado abaixo do preço justo.")

    if len(all_valued_df) > 0:
        print("\nMais próximas do preço justo:")
        near = all_valued_df.nlargest(10, 'margem_seg_media_pct')
        display(near[['tipo', 'ticker', 'nome', 'preco', 'preco_justo_dcf',
                       'preco_justo_graham', 'margem_seg_media_pct']])

🏆 TOP 20 SUBVALORIZADAS (ações + bancos)



,tipo,ticker,nome,setor,preco,preco_justo_dcf,preco_justo_graham,margem_seg_media_pct,forte_desconto,pl,pvp,roe_pct,margem_liquida_pct
0,ação,SAPR4,SANEPAR PN N2,Utilities,R$ 8.65,R$ 78.38,R$ 29.14,79.6%,True,6.71,0.21,17.9%,28.9%
1,ação,SAPR3,SANEPAR ON N2,Utilities,R$ 10.65,R$ 156.76,R$ 29.14,78.3%,True,8.26,0.26,17.9%,28.9%
2,ação,EALT4,ACO ALTONA PN,Industrials,R$ 12.96,R$ 113.40,R$ 30.74,73.2%,True,3.38,0.82,28.0%,16.7%
3,banco,BSLI4,BRB BANCO PN,Financial Services,R$ 3.90,R$ 16.49,R$ 12.15,72.1%,True,1.93,0.48,25.1%,19.2%
4,ação,ALUP4,ALUPAR PN N2,Utilities,R$ 10.94,R$ 53.28,R$ 23.57,66.5%,True,8.82,0.39,14.3%,27.6%
5,ação,CSED3,CRUZEIRO EDUON NM,Consumer Defensive,R$ 5.29,R$ 35.72,R$ 7.70,58.2%,True,6.53,1.21,19.4%,10.5%
6,ação,IGTI3,IGUATEMI S.AON N1,Real Estate,R$ 3.22,R$ 7.44,R$ 7.06,55.6%,True,6.85,0.21,13.0%,39.2%
7,ação,VTRU3,VITRUEDUCA ON NM,Consumer Defensive,R$ 14.31,R$ 32.78,R$ 30.58,54.8%,True,5.61,0.65,13.5%,16.4%
8,ação,JHSF3,JHSF PART ON NM,Real Estate,R$ 9.38,R$ 34.73,R$ 13.30,51.2%,True,3.39,0.99,21.0%,66.0%
9,ação,EUCA4,EUCATEX PN N1,Industrials,R$ 20.67,R$ 37.42,R$ 40.27,46.7%,True,5.97,0.69,12.0%,10.3%


## 7. Resumo da Análise

In [21]:
# Sumário
print("=" * 70)
print("RESUMO DA ANÁLISE")
print("=" * 70)
print(f"Total de tickers analisados:     {len(tickers_df)}")
print(f"\nAções que passaram nos filtros:   {len(filtered_stocks)}")
if len(valued_stocks) > 0:
    print(f"  Subvalorizadas (ações):        {valued_stocks['undervalued'].sum()}")
    print(f"  Forte desconto (≥20%):         {valued_stocks['forte_desconto'].sum()}")
print(f"\nBancos que passaram nos filtros:  {len(filtered_banks)}")
if len(valued_banks) > 0:
    print(f"  Subvalorizados (bancos):       {valued_banks['undervalued'].sum()}")
    print(f"  Forte desconto (≥20%):         {valued_banks['forte_desconto'].sum()}")
print(f"\nTop 20 selecionadas:             {len(top20)}")
print("=" * 70)
print(f"\nModelos de valuation:")
print(f"  Ações: DCF 2-estágios (decay linear, {valuation.PROJECTION_YEARS}y) + DDM fallback + Graham")
print(f"  Bancos: Excess Returns + Graham")
print(f"\nParâmetros: Selic={valuation.SELIC*100:.2f}%, "
      f"Crescimento terminal={valuation.TERMINAL_GROWTH*100:.1f}%, "
      f"Projeção={valuation.PROJECTION_YEARS} anos")
print(f"Margem de segurança mínima (forte desconto): {valuation.MIN_SAFETY_MARGIN_PCT:.0f}%")

RESUMO DA ANÁLISE
Total de tickers analisados:     388

Ações que passaram nos filtros:   27
  Subvalorizadas (ações):        19
  Forte desconto (≥20%):         16

Bancos que passaram nos filtros:  6
  Subvalorizados (bancos):       4
  Forte desconto (≥20%):         4

Top 20 selecionadas:             20

Modelos de valuation:
  Ações: DCF 2-estágios (decay linear, 10y) + DDM fallback + Graham
  Bancos: Excess Returns + Graham

Parâmetros: Selic=14.25%, Crescimento terminal=3.5%, Projeção=10 anos
Margem de segurança mínima (forte desconto): 20%
